# CEOAI Practice 1 - Star Observatory Minimum Solution

Objective: produce the first valid 600-row-style submission:

1. Estimate star centers with an intensity-weighted centroid.
2. Train a simple flux regressor from image summary features.
3. Export center and flux rows in the required CSV format.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
DATA = ROOT / "data"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

In [ ]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
print(train.head())
print({"train": len(train), "test": len(test)})

In [ ]:
def load_gray(path):
    return np.asarray(Image.open(path).convert("L"), dtype=float)

def image_features(image):
    yy, xx = np.mgrid[0:image.shape[0], 0:image.shape[1]]
    weights = np.maximum(image - np.percentile(image, 98), 0)
    total = weights.sum() + 1e-9
    cx = float((weights * xx).sum() / total)
    cy = float((weights * yy).sum() / total)
    return {
        "sum": float(image.sum()),
        "mean": float(image.mean()),
        "std": float(image.std()),
        "max": float(image.max()),
        "centroid_x": cx,
        "centroid_y": cy,
    }

def build_feature_frame(df, folder):
    rows = []
    for image_id in df["image_id"]:
        image = load_gray(DATA / folder / image_id)
        rows.append({"image_id": image_id, **image_features(image)})
    return pd.DataFrame(rows)

train_features = build_feature_frame(train, "train_images").merge(train, on="image_id")
test_features = build_feature_frame(test, "test_images")
train_features.head()

In [ ]:
feature_cols = ["sum", "mean", "std", "max", "centroid_x", "centroid_y"]
X_train, X_val, y_train, y_val = train_test_split(
    train_features[feature_cols],
    np.log1p(train_features["target_flux"]),
    test_size=0.25,
    random_state=0,
)
flux_model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
flux_model.fit(X_train, y_train)
val_pred = np.expm1(flux_model.predict(X_val))
print({"fixture_flux_rmse": float(np.sqrt(mean_squared_error(np.expm1(y_val), val_pred)))})

In [ ]:
test_flux = np.expm1(flux_model.predict(test_features[feature_cols]))
rows = []
for i, row in test_features.iterrows():
    image_id = row["image_id"]
    rows.append({
        "subtaskID": 1,
        "datapointID": image_id,
        "answer": f"({row['centroid_x']:.2f}, {row['centroid_y']:.2f})",
    })
    rows.append({
        "subtaskID": 2,
        "datapointID": image_id,
        "answer": float(test_flux[i]),
    })

submission = pd.DataFrame(rows)
submission.to_csv(OUT / "submission.csv", index=False)
assert len(submission) == 2 * len(test)
submission.head()

In [ ]:
hidden_path = DATA / "test_hidden.csv"
if hidden_path.exists():
    hidden = pd.read_csv(hidden_path).merge(test_features, on="image_id")
    center_mae = mean_absolute_error(hidden[["center_x", "center_y"]], hidden[["centroid_x", "centroid_y"]])
    flux_rmse = np.sqrt(mean_squared_error(hidden["target_flux"], test_flux))
    print({"fixture_center_coord_mae": float(center_mae), "fixture_flux_rmse": float(flux_rmse)})

print("wrote", OUT / "submission.csv")